# 06. Main-manuscript analysis

**Primary endpoint**: Success rate vs candidate budget k

4 methods:
1. Scaffold-tuner v7
2. Random scaffold-preserving local editing
3. CReM with scaffold protection
4. REINVENT4 LibInvent (task-directed RL)

**7 tasks** (AR −1 excluded): HBD +1, HBD −1, HBA +1, HBA −1, RotB +1, RotB −1, AR +1

Design:
- `candidate_rank` used as-is; no post-hoc shuffling
- Evaluation unit: parent × task (N = 1000 × 7 = 7000)
- Parents with 0 candidates → False in denominator (never inflated)
- Same success/strict-success definition for all methods

**Figures**:
- Figure 1: Success rate vs k (4 methods)
- Figure 2: Strict success rate vs k
- Figure 3: First-success rank / candidate efficiency


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import ACTIVE_PARENT_DATASET, TASKS, TASK_NAMES, K_VALUES
from utils.evaluation import success_at_k, success_rate_at_k, success_rate_by_task_at_k
from utils.records import compute_first_success_rank


In [ ]:
RESULTS_DIR = Path("../results/new_benchmark")
FIGURE_DIR  = RESULTS_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PARENT_FILE = Path(ACTIVE_PARENT_DATASET)

METHOD_FILES = {
    "Scaffold-tuner": RESULTS_DIR / "scaffold_tuner.csv",
    "Random":         RESULTS_DIR / "random_local_editing.csv",
    "CReM":           RESULTS_DIR / "crem.csv",
    "LibInvent (RL)": RESULTS_DIR / "libinvent.csv",
}
METHOD_ORDER = list(METHOD_FILES.keys())

parents = pd.read_csv(PARENT_FILE)
if "parent_id" not in parents.columns and "chembl_id" in parents.columns:
    parents = parents.rename(columns={"chembl_id": "parent_id"})
all_parent_ids = parents["parent_id"].unique()

results = {}
for method, path in METHOD_FILES.items():
    if path.exists():
        df = pd.read_csv(path)
        if "parent_id" not in df.columns and "chembl_id" in df.columns:
            df = df.rename(columns={"chembl_id": "parent_id"})
        results[method] = df
        print(f"  {method}: {len(df):,} records  tasks={df['task'].nunique()}")
    else:
        print(f"  {method}: NOT FOUND (skipped)")

methods_present = [m for m in METHOD_ORDER if m in results]
print(f"Parents: {len(all_parent_ids)}  Tasks: {len(TASK_NAMES)}  "
      f"Denominator: {len(all_parent_ids)*len(TASK_NAMES)}")


## Success@k

In [ ]:
success_rows = []
for method, df in results.items():
    for k in K_VALUES:
        s     = success_at_k(df, k, all_parent_ids, metric="success")
        s_str = success_at_k(df, k, all_parent_ids, metric="strict_success")
        success_rows.append({
            "method": method, "k": k,
            "success_rate":        s.mean(),
            "strict_success_rate": s_str.mean(),
        })

df_success = pd.DataFrame(success_rows)
pivot = df_success.pivot(index="k", columns="method", values="success_rate")[
    [m for m in METHOD_ORDER if m in df_success["method"].unique()]
]
print("Overall Success@k (%):")
print((pivot * 100).round(1).to_string())


## Figure 1: Success rate vs candidate budget k

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for method in methods_present:
    part = df_success[df_success["method"] == method].sort_values("k")
    ax.plot(part["k"], part["success_rate"] * 100,
            marker="o", linewidth=2, label=method)

ax.set_xscale("log")
ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES)
ax.set_xlabel("Candidate budget k")
ax.set_ylabel("Success rate across 7 tasks (%)")
ax.set_title("Figure 1: Success@k — 4 methods")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig1_success_at_k.pdf", dpi=300)
fig.savefig(FIGURE_DIR / "fig1_success_at_k.png", dpi=300)
plt.show()


## Figure 2: Strict success rate vs k

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for method in methods_present:
    part = df_success[df_success["method"] == method].sort_values("k")
    ax.plot(part["k"], part["strict_success_rate"] * 100,
            marker="s", linewidth=2, linestyle="--", label=method)

ax.set_xscale("log")
ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES)
ax.set_xlabel("Candidate budget k")
ax.set_ylabel("Strict success rate (%)")
ax.set_title("Figure 2: Strict Success@k (target Δ only; no collateral change)")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig2_strict_success_at_k.pdf", dpi=300)
fig.savefig(FIGURE_DIR / "fig2_strict_success_at_k.png", dpi=300)
plt.show()


## Figure 3: Candidate efficiency — first_success_rank

`first_success_rank` is the minimum `candidate_rank` at which a success candidate
first appears for a given parent × task pair.
Lower rank = more efficient (fewer candidates needed to find a success).

A parent × task with no success candidate has first_success_rank = NaN
(excluded from the boxplot but counted in denominator for Success@k).


In [ ]:
fsr_rows = []
for method, df in results.items():
    fsr = compute_first_success_rank(df, all_parent_ids)
    fsr["method"] = method
    fsr_rows.append(fsr)

df_fsr = pd.concat(fsr_rows, ignore_index=True)
df_fsr_success = df_fsr.dropna(subset=["first_success_rank"])

# Boxplot
fig, ax = plt.subplots(figsize=(7, 5))
data_plot = [
    df_fsr_success[df_fsr_success["method"] == m]["first_success_rank"].values
    for m in methods_present
]
bp = ax.boxplot(data_plot, tick_labels=methods_present, showfliers=False, patch_artist=True)
ax.set_ylabel("First success rank")
ax.set_title("Figure 3: Candidate efficiency (first_success_rank among succeeded pairs)")
ax.set_yscale("log")
ax.set_yticks([1, 5, 10, 20, 50, 100])
ax.set_yticklabels([1, 5, 10, 20, 50, 100])
ax.grid(True, alpha=0.3, axis="y")
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "fig3_first_success_rank.pdf", dpi=300)
fig.savefig(FIGURE_DIR / "fig3_first_success_rank.png", dpi=300)
plt.show()

# Median summary
print("Median first_success_rank (among succeeded parent×task pairs):")
print(df_fsr_success.groupby("method")["first_success_rank"].median().reindex(methods_present).round(1))


## Summary table (k = 1, 5, 10, 20, 50, 100)

In [ ]:
rows = []
for method, df in results.items():
    for k in K_VALUES:
        s     = success_at_k(df, k, all_parent_ids, "success").mean()
        s_str = success_at_k(df, k, all_parent_ids, "strict_success").mean()
        rows.append({"method": method, "k": k,
                     "success_%": round(s*100, 1), "strict_%": round(s_str*100, 1)})

df_tbl = pd.DataFrame(rows)
display(df_tbl.pivot_table(index="method", columns="k",
                            values=["success_%","strict_%"]).reindex(methods_present))
